# Laboratorio · Sesión 3 — Calidad de datos: perfilado, reglas y métricas
**PY-EAN · Python para Procesamiento, Calidad y Automatización de Datos** · Universidad Ean

**Caso:** la gerencia pregunta *"¿podemos confiar en la base de afiliados?"*. No basta con "se ve bien":
vas a responder con un **reporte de calidad** — % de cumplimiento por regla, por dimensión y un índice global —
y a separar los registros válidos de los que van a **cuarentena**.

**Cómo trabajar:** Shift + Enter de arriba hacia abajo · completa donde dice `# ✏️ TU CÓDIGO AQUÍ` ·
guarda tu copia en Drive (**Archivo → Guardar una copia en Drive**).

> Todos los datos son sintéticos. No corresponden a personas reales.

## Parte 0 · Preparar el entorno (5 min)

In [ ]:
# ▶️ Ejecuta esta celda primero (Shift + Enter)
import os
import pandas as pd

URL_DATOS = "https://raw.githubusercontent.com/Hackwy402/python-procesamiento-ean/main/datos/"
# Si subiste la carpeta "datos" a Colab (o trabajas en tu PC) se usa la copia local;
# si no, se leen los datos directamente desde GitHub.
for carpeta in ["datos/", "../datos/", "/content/datos/"]:
    if os.path.exists(carpeta + "afiliados.csv"):
        RUTA = carpeta
        break
else:
    RUTA = URL_DATOS

pd.set_option("display.max_columns", 30)
print("Leyendo datos desde:", RUTA)

In [ ]:
import re
import unicodedata
import numpy as np
import matplotlib.pyplot as plt

HOY = pd.Timestamp("2026-10-31")     # fecha de corte del caso
SMMLV = 1_750_905                    # salario mínimo 2026 (dato de referencia del curso)

# dtype=str: documentos y teléfonos son TEXTO (no queremos que pandas los vuelva números)
afiliados = pd.read_csv(RUTA + "afiliados.csv", dtype={"numero_documento": str, "telefono": str})
cat_ciudades = pd.read_excel(RUTA + "oficinas.xlsx", sheet_name="ciudades", dtype={"codigo_dane": str})
pqrs = pd.read_csv(RUTA + "pqrs.csv")
aportes = pd.read_csv(RUTA + "aportes/aportes_2026-03.csv")

print("afiliados:", afiliados.shape, "| pqrs:", pqrs.shape, "| aportes marzo:", aportes.shape)

## Parte 1 · Perfilado (15 min)
Antes de escribir reglas, mira qué hay.

**✏️ Ejercicio 1.** Calcula el **porcentaje** de vacíos por columna de `afiliados`, redondeado a 2 decimales y ordenado
de mayor a menor. Pista: `isna().mean() * 100`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 2.** ¿Cuántos valores distintos tiene cada columna (`nunique`)? Compara `id_afiliado` y
`numero_documento` contra el número de filas. ¿Qué te dice eso?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 3.** Completa la función `perfilar(df)` para que devuelva **una fila por columna** con:
`tipo`, `pct_nulos`, `distintos` y `ejemplo` (el primer valor no vacío).

In [ ]:
def perfilar(df):
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "pct_nulos": None,   # ✏️ % de nulos por columna, 2 decimales
        "distintos": None,   # ✏️ valores distintos por columna
        "ejemplo": df.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })

perfilar(afiliados)

**✏️ Ejercicio 4.** Reutiliza tu función: perfila `pqrs` y `aportes`. ¿Qué columnas tienen vacíos? ¿Tiene sentido que los tengan?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

✍️ *Tu respuesta:* (pista: ¿una PQRS **abierta** debería tener fecha de respuesta?)

## Parte 2 · Reglas como máscaras booleanas (20 min)
Una regla es una pregunta de **sí/no por registro**: `True` = cumple, `False` = falla.
El promedio de la máscara es el % de cumplimiento.

In [ ]:
# Ejemplo resuelto — Validez de dominio: el estado debe ser uno de los valores permitidos
ok_estado = afiliados["estado"].isin(["Activo", "Inactivo", "Pensionado"])
print("Fallas:", (~ok_estado).sum(), "| Cumplimiento:", round(ok_estado.mean() * 100, 2), "%")
afiliados.loc[~ok_estado, "estado"].value_counts()

**✏️ Ejercicio 5 · Validez.** El `numero_documento` debe tener **solo dígitos, entre 6 y 10**.
Crea `ok_doc` con `.str.match(r"^\d{6,10}$", na=False)`, cuenta las fallas y muestra 10 ejemplos que fallan.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 6 · Validez.** Crea `ok_email` (patrón `PATRON_EMAIL`) y `ok_tel` (celular: 10 dígitos que empieza
por 3 → `r"^3\d{9}$"`). Cuenta las fallas de cada una.

In [ ]:
PATRON_EMAIL = r"^[\w.+-]+@[\w-]+\.[\w.]+$"

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 7 · Exactitud (plausibilidad).** Las fechas vienen en dos formatos. Usa la función `a_fecha` para
convertir `fecha_nacimiento`, calcula la `edad` al `HOY` y crea `ok_edad` = edad entre 18 y 100.
¿Cuántas fechas **no se pudieron leer**? ¿Cuántas edades son imposibles?

In [ ]:
def a_fecha(serie):
    """Convierte texto a fecha aceptando 1971-09-13 y 13/09/1971."""
    iso = pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce")
    return iso.fillna(dmy)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 8 · Exactitud.** Ningún salario base puede ser inferior al SMMLV. Convierte `salario_base` a número
(`pd.to_numeric(..., errors="coerce")`) y crea `ok_salario`. Separa las fallas en tres grupos:
**no numérico**, **negativo** y **positivo pero bajo el mínimo**.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 9 · Consistencia.** El `departamento` debe corresponder a la `ciudad` según el catálogo.
Completa: usa `mapa` para obtener el departamento esperado; la regla **solo aplica** si la ciudad está en el catálogo
(si no está, la regla "no aplica" y cuenta como cumple).

In [ ]:
mapa = dict(zip(cat_ciudades["ciudad"], cat_ciudades["departamento"]))
esperado = None   # ✏️ departamento esperado para cada afiliado (usa .map)
ok_depto = None   # ✏️ esperado vacío  O  esperado == departamento

**✏️ Ejercicio 10 · Consistencia (regla de negocio).** Un afiliado con estado *pensionado* debe tener la edad mínima:
**57 mujeres / 62 hombres**. Completa `ok_pension` (los no pensionados cumplen automáticamente).

In [ ]:
g = afiliados["genero"].str.strip().str.upper().str[0]          # F / M (normalizado)
es_pens = afiliados["estado"].str.strip().str.lower().str.startswith("pensionad")
edad_min = pd.Series(np.where(g == "F", 57, 62), index=afiliados.index)
ok_pension = None   # ✏️ no es pensionado  O  edad >= edad_min

## Parte 3 · Catálogo de reglas, motor y reporte (20 min)
Las reglas sueltas no escalan. Las ponemos en una **lista de diccionarios** y un **motor** las aplica todas.

In [ ]:
def _edad(df):
    return (HOY - a_fecha(df["fecha_nacimiento"])).dt.days // 365.25

def _ok_pension(df):
    g = df["genero"].str.strip().str.upper().str[0]
    es_pens = df["estado"].str.strip().str.lower().str.startswith("pensionad")
    minimo = pd.Series(np.where(g == "F", 57, 62), index=df.index)
    return ~es_pens | (_edad(df) >= minimo)

def _ok_afiliacion(df):
    nac, afi = a_fecha(df["fecha_nacimiento"]), a_fecha(df["fecha_afiliacion"])
    return nac.isna() | (afi >= nac + pd.DateOffset(years=18))

def _ok_depto(df):
    esperado = df["ciudad"].map(dict(zip(cat_ciudades["ciudad"], cat_ciudades["departamento"])))
    return esperado.isna() | (esperado == df["departamento"])

REGLAS_AFILIADOS = [
    {"nombre": "documento_presente", "dimension": "Completitud", "severidad": "crítica",
     "valida": lambda df: df["numero_documento"].notna()},
    {"nombre": "genero_presente", "dimension": "Completitud", "severidad": "advertencia",
     "valida": lambda df: df["genero"].notna()},
    {"nombre": "fila_no_duplicada", "dimension": "Unicidad", "severidad": "crítica",
     "valida": lambda df: ~df.duplicated()},
    {"nombre": "documento_unico", "dimension": "Unicidad", "severidad": "crítica",
     "valida": lambda df: ~df.duplicated(subset=["numero_documento"])},
    {"nombre": "documento_formato", "dimension": "Validez", "severidad": "crítica",
     "valida": lambda df: df["numero_documento"].str.match(r"^\d{6,10}$", na=False)},
    {"nombre": "email_formato", "dimension": "Validez", "severidad": "advertencia",
     "valida": lambda df: df["email"].str.match(PATRON_EMAIL, na=False)},
    {"nombre": "telefono_celular", "dimension": "Validez", "severidad": "advertencia",
     "valida": lambda df: df["telefono"].str.match(r"^3\d{9}$", na=False)},
    {"nombre": "genero_dominio", "dimension": "Validez", "severidad": "advertencia",
     "valida": lambda df: df["genero"].isin(["F", "M"])},
    {"nombre": "estado_dominio", "dimension": "Validez", "severidad": "advertencia",
     "valida": lambda df: df["estado"].isin(["Activo", "Inactivo", "Pensionado"])},
    {"nombre": "ciudad_en_catalogo", "dimension": "Validez", "severidad": "advertencia",
     "valida": lambda df: df["ciudad"].isin(cat_ciudades["ciudad"])},
    {"nombre": "edad_18_100", "dimension": "Exactitud", "severidad": "crítica",
     "valida": lambda df: _edad(df).between(18, 100)},
    {"nombre": "depto_coherente", "dimension": "Consistencia", "severidad": "advertencia",
     "valida": _ok_depto},
    {"nombre": "afiliacion_mayor_edad", "dimension": "Consistencia", "severidad": "crítica",
     "valida": _ok_afiliacion},
    {"nombre": "pensionado_edad_minima", "dimension": "Consistencia", "severidad": "crítica",
     "valida": _ok_pension},
]
print("Reglas en el catálogo:", len(REGLAS_AFILIADOS))

**✏️ Ejercicio 11.** Agrega al catálogo la regla **`salario_minimo`** (dimensión *Exactitud*, severidad *crítica*):
el salario numérico debe ser ≥ SMMLV.

In [ ]:
REGLAS_AFILIADOS.append(
    # ✏️ {"nombre": ..., "dimension": ..., "severidad": ..., "valida": lambda df: ...}
)
print("Reglas en el catálogo:", len(REGLAS_AFILIADOS))

### El motor
Recibe una tabla y un catálogo; devuelve el **reporte**, la columna de **errores** por registro y una marca de
**falla crítica**. Léelo con calma: es solo un `for` sobre la lista de reglas.

In [ ]:
def evaluar(df, reglas):
    filas = []
    errores = pd.Series("", index=df.index)
    critica = pd.Series(False, index=df.index)
    for r in reglas:
        ok = pd.Series(r["valida"](df), index=df.index).fillna(False).astype(bool)
        filas.append({"regla": r["nombre"], "dimension": r["dimension"], "severidad": r["severidad"],
                      "evaluados": len(df), "fallas": int((~ok).sum()),
                      "cumplimiento_%": round(ok.mean() * 100, 2)})
        errores = errores.where(ok, errores + r["nombre"] + "; ")
        if r["severidad"] == "crítica":
            critica = critica | ~ok
    return pd.DataFrame(filas), errores.str.rstrip("; "), critica

def semaforo(p):
    return "🟢" if p >= 98 else ("🟡" if p >= 95 else "🔴")

**✏️ Ejercicio 12.** Ejecuta el motor sobre `afiliados` y muestra el reporte ordenado de **menor a mayor** cumplimiento, con una columna `semaforo`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 13.** Calcula el **índice por dimensión** (promedio del cumplimiento de sus reglas) y el **índice global**
(promedio de las dimensiones). Luego grafica el índice por dimensión en barras horizontales.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 14 · Cuarentena.** Agrega a una copia de `afiliados` la columna `errores` y separa:
`validos` (sin falla crítica) y `cuarentena` (con al menos una). ¿Cuántos registros hay en cada una?
¿Cuáles son las combinaciones de errores más frecuentes en cuarentena?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

## Parte 4 · El mismo motor, otras tablas (10 min)
**✏️ Ejercicio 15 · PQRS.** Completa el catálogo de PQRS con la regla de **oportunidad**: respuesta en máximo
**15 días hábiles** (`np.busday_count`). Ejecuta el motor y muestra el reporte.

In [ ]:
def dias_habiles(df):
    rad, resp = pd.to_datetime(df["fecha_radicacion"]), pd.to_datetime(df["fecha_respuesta"])
    m = resp.notna()
    dias = pd.Series(np.nan, index=df.index)
    dias[m] = np.busday_count(rad[m].values.astype("datetime64[D]"), resp[m].values.astype("datetime64[D]"))
    return dias

REGLAS_PQRS = [
    {"nombre": "radicado_unico", "dimension": "Unicidad", "severidad": "crítica",
     "valida": lambda df: ~df["radicado"].duplicated()},
    {"nombre": "cerrado_con_fecha", "dimension": "Completitud", "severidad": "crítica",
     "valida": lambda df: (df["estado"] != "Cerrado") | df["fecha_respuesta"].notna()},
    {"nombre": "respuesta_despues_radicacion", "dimension": "Consistencia", "severidad": "crítica",
     "valida": lambda df: df["fecha_respuesta"].isna()
                          | (pd.to_datetime(df["fecha_respuesta"]) >= pd.to_datetime(df["fecha_radicacion"]))},
]

In [ ]:
REGLAS_PQRS.append(
    # ✏️ regla "respuesta_15_dias_habiles" (Oportunidad, advertencia): sin respuesta  O  dias_habiles <= 15
)
rep_pqrs, err_pqrs, crit_pqrs = evaluar(pqrs, REGLAS_PQRS)
rep_pqrs

**✏️ Ejercicio 16 · Aportes.** Crea `REGLAS_APORTES` con al menos estas tres reglas y evalúa los aportes de marzo:
- `aporte_16_pct` (Exactitud): `|valor_aporte − round(ibc × 0.16)| ≤ 1`
- `ibc_minimo` (Exactitud): `ibc ≥ SMMLV`
- `afiliado_existe` (Integridad referencial): el `id_afiliado` está en `afiliados`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

## Parte 5 · Medir → limpiar → volver a medir (5 min)
La función `limpiar_minimo` aplica lo de la Sesión 2 (versión corta). Ejecútala y compara el índice **antes y después**.

In [ ]:
def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def clave_ciudad(s):
    return sin_tildes(str(s)).lower().strip().replace(".", "").replace(" dc", "")

EXTRA_CIUDADES = {"santiago de cali": "Cali", "b/quilla": "Barranquilla",
                  "cartagena de indias": "Cartagena", "b/manga": "Bucaramanga"}
MAPA_CIUDAD = {clave_ciudad(c): c for c in cat_ciudades["ciudad"]} | EXTRA_CIUDADES

def limpiar_minimo(df):
    df = df.drop_duplicates().copy()
    df["numero_documento"] = df["numero_documento"].str.replace(r"\D", "", regex=True)
    df["nombres"] = df["nombres"].str.strip().str.title()
    df["apellidos"] = df["apellidos"].str.strip().str.title()
    df["genero"] = df["genero"].str.strip().str.upper().str[0]
    df["estado"] = (df["estado"].str.strip().str.capitalize()
                    .replace({"Pensionada": "Pensionado"}))
    claves = df["ciudad"].map(clave_ciudad)
    df["ciudad"] = claves.map(MAPA_CIUDAD).fillna(df["ciudad"])
    df["email"] = df["email"].str.strip().str.lower()
    df["salario_base"] = pd.to_numeric(df["salario_base"].str.replace(".", "", regex=False), errors="coerce")
    return df

**✏️ Ejercicio 17.** Limpia, vuelve a evaluar y arma una tabla comparativa del índice por dimensión **antes vs. después**. ¿Qué dimensiones mejoraron? ¿Cuáles **no** y por qué?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

✍️ *Tu respuesta:* (pista: ¿la limpieza puede inventar una fecha de nacimiento correcta o el salario real de alguien?)

## 🎯 Reto de cierre
1. Agrega a `REGLAS_APORTES` la regla **`pago_oportuno`** (Oportunidad): la `fecha_pago` debe ser a más tardar el
   **último día del mes siguiente** al `periodo`. Pista: `pd.to_datetime(df["periodo"]) + pd.offsets.MonthEnd(2)`.
2. Escribe una regla propia para `afiliados` que no hayamos visto y explica a qué dimensión pertenece.
3. **Para la gerencia:** en 3 líneas, ¿se puede confiar en la base de afiliados? Usa el índice global y las 2 reglas
   más críticas como evidencia.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ